In [241]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent

if not (project_root / "src").is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục src từ: {cwd}")

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print(f"Project root: {project_root}")

Project root: /home/trieu/Intern/SwM_precomputed


In [242]:
from src.data.train_dataset import TrainDataset
from torch.utils.data import DataLoader

In [243]:
train_samples = [
    {
        'history': [
            'N8129',
            'N1569',
            'N17686',
        ],
        'target': 'N13008',
    },
    {
        'history': [
            'N63302',
            'N10414',
            'N19347',
            'N31801'
        ],
        'target': 'N55689'
    },
    {
        'history': [
            'N21623',
            'N6233',
            'N14340',
            'N48031',
            'N62285'
        ],
        'target': 'N31739'
    },
    {
        'history': [
            'N31739',
            'N6072',
            'N63045',
            'N23979',
            'N35656',
        ],
        'target': 'N43353'
    },
    
]

In [244]:
mapping = load_pickle(project_root / "data/processed/mindsmall_v2/artifacts/news_vector_mapping.pkl")
train_dataset = TrainDataset(samples=train_samples, max_sequence_length=5, padding_id=0, mapping=mapping, vector_size=384)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

In [245]:
batch = next(iter(train_loader))

In [246]:
from torch import nn
import torch

In [247]:
positional_embedding = nn.Embedding(5, 384)
position_vectors = torch.arange(0, 5).unsqueeze(0)
embedded_positions = positional_embedding(position_vectors)
embedded_positions.shape

torch.Size([1, 5, 384])

In [248]:
dropout = nn.Dropout(p=0.2)

In [249]:
input_vectors = batch['input_vectors']
embeddings = dropout(input_vectors + embedded_positions)
embeddings.shape

torch.Size([2, 5, 384])

In [250]:
input_vectors

tensor([[[ 0.0474,  0.0235, -0.0266,  ...,  0.0044, -0.0039, -0.0497],
         [-0.0393,  0.0446, -0.0069,  ..., -0.0311, -0.0405,  0.0414],
         [-0.0224,  0.0937, -0.0662,  ...,  0.0277,  0.0182,  0.0180],
         [-0.0197, -0.0263,  0.0338,  ..., -0.0918,  0.0355,  0.0769],
         [ 0.0133,  0.0343, -0.0055,  ...,  0.0624,  0.0234, -0.0187]],

        [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [-0.0468,  0.0341,  0.0835,  ..., -0.0013, -0.0461,  0.0131],
         [ 0.0014, -0.0513, -0.0381,  ..., -0.0201,  0.0123, -0.0144],
         [ 0.0309, -0.0144, -0.1173,  ..., -0.0786, -0.0134, -0.0172],
         [ 0.0139,  0.0782,  0.0227,  ..., -0.0255, -0.0657, -0.0063]]])

In [251]:
embeddings

tensor([[[-1.2617, -0.0000, -0.0000,  ..., -3.2629,  0.0000,  0.8244],
         [-0.3305, -0.2359, -0.1178,  ...,  0.0106, -0.6907, -0.5194],
         [ 1.2205,  1.4596, -1.2522,  ...,  0.8578, -0.0000,  0.3144],
         [ 0.3324,  0.1419,  1.7684,  ...,  0.6197,  0.9942, -0.4482],
         [ 2.3752, -1.2407, -0.2393,  ..., -0.5442, -0.7755, -1.3654]],

        [[-1.3210, -0.5738, -0.2166,  ..., -3.2684,  0.0000,  0.8865],
         [-0.3399, -0.2489, -0.0048,  ...,  0.0479, -0.6978, -0.5548],
         [ 1.2503,  1.2784, -1.2170,  ...,  0.7980, -1.3084,  0.2739],
         [ 0.3957,  0.1568,  1.5795,  ...,  0.0000,  0.9330, -0.5657],
         [ 2.3759, -0.0000, -0.2040,  ..., -0.6541, -0.8869, -1.3498]]],
       grad_fn=<MulBackward0>)

In [252]:
from src.models.sasrec import sequence_padding_mask, create_attention_mask

In [253]:
padding_mask = sequence_padding_mask(input_vectors)
padding_mask, padding_mask.shape

(tensor([[ True,  True,  True,  True,  True],
         [False,  True,  True,  True,  True]]),
 torch.Size([2, 5]))

In [254]:
attention_mask = create_attention_mask(input_vectors)
attention_mask, attention_mask.shape

(tensor([[[ True, False, False, False, False],
          [ True,  True, False, False, False],
          [ True,  True,  True, False, False],
          [ True,  True,  True,  True, False],
          [ True,  True,  True,  True,  True]],
 
         [[False, False, False, False, False],
          [False,  True, False, False, False],
          [False,  True,  True, False, False],
          [False,  True,  True,  True, False],
          [False,  True,  True,  True,  True]]]),
 torch.Size([2, 5, 5]))